In [ ]:
import time
import os
from datetime import datetime
from typing import Dict, Any, List, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from collections import deque, namedtuple, Counter
import gymnasium as gym

from bsk_rl import ConstellationTasking
from bsk_rl.sats import ImagingSatellite
from bsk_rl.act import Action, Image
from bsk_rl import obs
from bsk_rl.sim import dyn, fsw
from bsk_rl.scene.targets import UniformTargets, Target
from bsk_rl.data import UniqueImageReward
from bsk_rl.comm import LOSCommunication
from bsk_rl.utils.orbital import walker_delta_args

Experience = namedtuple('Experience', 
                       ['state', 'action', 'reward', 'next_state', 'done'])

class HindsightExperienceReplay:
    def __init__(self, capacity=100000, her_ratio=0.8):
        self.capacity = capacity
        self.her_ratio = her_ratio
        self.buffer = deque(maxlen=capacity)
        self.episode_buffer = []
        
    def add(self, state, action, reward, next_state, done):
        self.buffer.append(Experience(state, action, reward, next_state, done))
        
    def sample(self, batch_size):
        batch = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        states = torch.FloatTensor(np.array([e.state for e in batch]))
        actions = torch.FloatTensor(np.array([e.action for e in batch]))
        rewards = torch.FloatTensor(np.array([e.reward for e in batch])).unsqueeze(1)
        next_states = torch.FloatTensor(np.array([e.next_state for e in batch]))
        dones = torch.FloatTensor(np.array([e.done for e in batch])).unsqueeze(1)
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.buffer)

class Actor(nn.Module):
    def __init__(self, state_dim, action_dims, hidden_dim=256):
        super(Actor, self).__init__()
        self.action_dims = action_dims
        self.total_actions = sum(action_dims)
        
        self.l1 = nn.Linear(state_dim, hidden_dim)
        self.l2 = nn.Linear(hidden_dim, hidden_dim)
        self.l3 = nn.Linear(hidden_dim, self.total_actions)
        
        nn.init.xavier_uniform_(self.l1.weight)
        nn.init.xavier_uniform_(self.l2.weight)
        nn.init.xavier_uniform_(self.l3.weight)
    
    def forward(self, state):
        x = F.relu(self.l1(state))
        x = F.relu(self.l2(x))
        x = self.l3(x)
        
        actions = []
        start_idx = 0
        for dim in self.action_dims:
            action_logits = x[:, start_idx:start_idx + dim]
            actions.append(F.softmax(action_logits, dim=-1))
            start_idx += dim
        
        return torch.cat(actions, dim=-1)

class Critic(nn.Module):
    def __init__(self, state_dim, action_dims, hidden_dim=256):
        super(Critic, self).__init__()
        self.total_actions = sum(action_dims)
        
        self.l1 = nn.Linear(state_dim + self.total_actions, hidden_dim)
        self.l2 = nn.Linear(hidden_dim, hidden_dim)
        self.l3 = nn.Linear(hidden_dim, 1)
        
        self.l4 = nn.Linear(state_dim + self.total_actions, hidden_dim)
        self.l5 = nn.Linear(hidden_dim, hidden_dim)
        self.l6 = nn.Linear(hidden_dim, 1)
        
        for layer in [self.l1, self.l2, self.l3, self.l4, self.l5, self.l6]:
            nn.init.xavier_uniform_(layer.weight)
    
    def forward(self, state, action):
        sa = torch.cat([state, action], 1)
        
        q1 = F.relu(self.l1(sa))
        q1 = F.relu(self.l2(q1))
        q1 = self.l3(q1)
        
        q2 = F.relu(self.l4(sa))
        q2 = F.relu(self.l5(q2))
        q2 = self.l6(q2)
        
        return q1, q2
    
    def Q1(self, state, action):
        sa = torch.cat([state, action], 1)
        q1 = F.relu(self.l1(sa))
        q1 = F.relu(self.l2(q1))
        q1 = self.l3(q1)
        return q1

class TD3_HER_DWC:
    def __init__(
        self,
        state_dim,
        action_dims,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        discount=0.99,
        tau=0.005,
        policy_freq=2,
        her_ratio=0.8,
        lr_actor=3e-4,
        lr_critic=3e-4,
    ):
        self.device = device
        self.discount = discount
        self.tau = tau
        self.policy_freq = policy_freq
        self.action_dims = action_dims
        self.total_actions = sum(action_dims)
        
        self.actor = Actor(state_dim, action_dims).to(device)
        self.actor_target = Actor(state_dim, action_dims).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=lr_actor)
        
        self.critic = Critic(state_dim, action_dims).to(device)
        self.critic_target = Critic(state_dim, action_dims).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        self.replay_buffer = HindsightExperienceReplay(her_ratio=her_ratio)
        
        self.total_iterations = 0
    
    def select_action(self, state, explore=True):
        state = torch.FloatTensor(state.reshape(1, -1)).to(self.device)
        action_probs = self.actor(state).cpu().data.numpy().flatten()
        
        actions = []
        start_idx = 0
        for dim in self.action_dims:
            probs = action_probs[start_idx:start_idx + dim]
            
            if explore:
                noise = np.random.dirichlet(np.ones(dim) * 0.1)
                probs = 0.9 * probs + 0.1 * noise
                probs = probs / np.sum(probs)  
            
            action = np.random.choice(dim, p=probs)
            actions.append(action)
            start_idx += dim
        
        return actions
    
    def dimension_wise_clipping(self, action_probs, original_probs, epsilon=0.1):
        """Apply dimension-wise clipping to action probabilities"""
        clipped_probs = []
        start_idx = 0
        for i, dim in enumerate(self.action_dims):
            original_dim_probs = original_probs[start_idx:start_idx + dim]
            current_dim_probs = action_probs[start_idx:start_idx + dim]
            
            clipped_dim_probs = np.clip(
                current_dim_probs,
                original_dim_probs - epsilon,
                original_dim_probs + epsilon
            )
            clipped_dim_probs = clipped_dim_probs / np.sum(clipped_dim_probs)  
            clipped_probs.extend(clipped_dim_probs)
            start_idx += dim
        
        return np.array(clipped_probs)
    
    def train(self, batch_size=256):
        if len(self.replay_buffer) < batch_size:
            return
        
        self.total_iterations += 1
        
        state, action, reward, next_state, done = self.replay_buffer.sample(batch_size)
        state = state.to(self.device)
        action = action.to(self.device)
        reward = reward.to(self.device)
        next_state = next_state.to(self.device)
        done = done.to(self.device)
        
        action_one_hot = self._actions_to_one_hot(action, batch_size)
        
        with torch.no_grad():
            next_action_probs = self.actor_target(next_state)
            
            noise = torch.clamp(torch.randn_like(next_action_probs) * 0.1, -0.2, 0.2)
            next_action_probs = torch.clamp(next_action_probs + noise, 1e-6, 1.0)
            next_action_probs = next_action_probs / next_action_probs.sum(dim=1, keepdim=True)
            
            target_Q1, target_Q2 = self.critic_target(next_state, next_action_probs)
            target_Q = torch.min(target_Q1, target_Q2)
            target_Q = reward + (1 - done) * self.discount * target_Q
        
        current_Q1, current_Q2 = self.critic(state, action_one_hot)
        
        critic_loss = F.mse_loss(current_Q1, target_Q) + F.mse_loss(current_Q2, target_Q)
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_optimizer.step()
        
        if self.total_iterations % self.policy_freq == 0:
            action_probs = self.actor(state)
            actor_loss = -self.critic.Q1(state, action_probs).mean()
            
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
            self.actor_optimizer.step()
            
            for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
            
            for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
    
    def _actions_to_one_hot(self, actions, batch_size):
        """Convert discrete actions to one-hot encoding"""
        one_hot_actions = []
        for i in range(batch_size):
            one_hot = []
            action_idx = 0
            for dim in self.action_dims:
                action_val = int(actions[i, action_idx].item())
                action_one_hot = torch.zeros(dim)
                action_one_hot[action_val] = 1.0
                one_hot.append(action_one_hot)
                action_idx += 1
            one_hot_actions.append(torch.cat(one_hot))
        
        return torch.stack(one_hot_actions).to(self.device)
    
    def _flatten_actions(self, actions):
        """Flatten list of actions into tensor"""
        return torch.FloatTensor(actions)
    
    def save(self, filename):
        torch.save({
            'actor': self.actor.state_dict(),
            'critic': self.critic.state_dict(),
            'actor_target': self.actor_target.state_dict(),
            'critic_target': self.critic_target.state_dict(),
            'actor_optimizer': self.actor_optimizer.state_dict(),
            'critic_optimizer': self.critic_optimizer.state_dict(),
        }, filename)
    
    def load(self, filename):
        checkpoint = torch.load(filename)
        self.actor.load_state_dict(checkpoint['actor'])
        self.critic.load_state_dict(checkpoint['critic'])
        self.actor_target.load_state_dict(checkpoint['actor_target'])
        self.critic_target.load_state_dict(checkpoint['critic_target'])
        self.actor_optimizer.load_state_dict(checkpoint['actor_optimizer'])
        self.critic_optimizer.load_state_dict(checkpoint['critic_optimizer'])

class TargetAreas(UniformTargets):
    def __init__(self, n_targets: int = 40, priority_distribution=None, radius=6378136.6):
        super().__init__(n_targets, priority_distribution, radius)
        self.lat_min, self.lat_max = -38.0, -25.0
        self.lon_min, self.lon_max = 129.0, 141.0

    def regenerate_targets(self) -> None:
        n_targets = self.n_targets if isinstance(self.n_targets, int) else np.random.randint(
            self.n_targets[0], self.n_targets[1] + 1
        )
        lats = np.random.uniform(self.lat_min, self.lat_max, n_targets)
        lons = np.random.uniform(self.lon_min, self.lon_max, n_targets)
        r = self.radius
        targets = []
        for i, (lat, lon) in enumerate(zip(lats, lons)):
            lat_rad = np.radians(lat)
            lon_rad = np.radians(lon)
            x = r * np.cos(lat_rad) * np.cos(lon_rad)
            y = r * np.cos(lat_rad) * np.sin(lon_rad)
            z = r * np.sin(lat_rad)
            priority = self.priority_distribution() if self.priority_distribution else np.random.uniform(0, 1)
            targets.append(Target(f"SA_Target_{i}", [x, y, z], priority))
        self.targets = targets

class Reallocate(Action):
    def __init__(self, n_sats=4):
        self.action_space = gym.spaces.Discrete(n_sats)
        self.n_actions = self.action_space.n
        self.option = None

    @property
    def builder_type(self):
        return Image.builder_type if isinstance(Image.builder_type, type) else ActionBuilder

    def set_action(self, option: int, **kwargs):
        self.option = option

    def action(self, satellite, state):
        return 0.0

class AdvancedImagingSatellite(ImagingSatellite):
    observation_spec = [
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="opportunity_open", norm=5700.0),
            n_ahead_observe=5,
        )
    ]
    action_spec = [Image(n_ahead_image=5), Reallocate()]
    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = fsw.SteeringImagerFSWModel

def _tune_access_generation(sat, initial=1800.0, step=120.0, max_dur=3600.0):
    candidates = [
        getattr(getattr(sat, "fsw", None), "opportunity_generator", None),
        getattr(getattr(sat, "dynamics", None), "accessGenerator", None),
        getattr(getattr(sat, "dynamics", None), "opportunityGenerator", None),
    ]
    og = next((c for c in candidates if c is not None), None)
    if og is None:
        return

    for name, val in [
        ("initial_generation_duration", float(initial)),
        ("generation_step", float(step)),
        ("max_generation_duration", float(max_dur)),
        ("max_lookahead", float(max_dur)),
    ]:
        if hasattr(og, name):
            setattr(og, name, val)

    extras = {"retask_on_image_complete": True, "max_access_compute_time_s": 2.0}
    for k, v in extras.items():
        if hasattr(og, k):
            setattr(og, k, v)

class CustomUniqueImageReward(UniqueImageReward):
    DEBUG_PROBE = False
    DEBUG_PROBE_STEPS = 5

    def __init__(self):
        try:
            super().__init__(data_store_kwargs={"keys": ["imaged", "image", "image_complete"]})
        except TypeError:
            super().__init__()
        self.imaged_by_sat = {f"Sat-{i}": 0 for i in range(4)}
        self._probe_count = 0
        self.imaged_targets = set()

    def reward(self, new_data_dict):
        all_step_targets = []
        for data in new_data_dict.values():
            imgs = getattr(data, "imaged", []) or []
            all_step_targets.extend(imgs)
        occ = Counter(all_step_targets)

        rewards = {}
        for sat_id, data in new_data_dict.items():
            total = 0.0
            new_unique_count = 0
            for tgt in getattr(data, "imaged", []) or []:
                if tgt not in self.data.imaged:
                    prio = float(getattr(tgt, "priority", 0.0))
                    denom = occ.get(tgt, 1) or 1
                    total += self.reward_fn(prio) / denom
                    new_unique_count += 1
            rewards[sat_id] = total
            if new_unique_count > 0:
                self.imaged_by_sat[sat_id] = self.imaged_by_sat.get(sat_id, 0) + new_unique_count

        for sat_id in getattr(self, "imaged_by_sat", {}).keys():
            rewards.setdefault(sat_id, 0.0)

        return rewards

class CustomConstellationTasking(ConstellationTasking):
    def __init__(self, *args, max_episode_steps=64, **kwargs):
        super().__init__(*args, **kwargs)
        self.max_episode_steps = int(max_episode_steps)
        self._step_count = 0
        self.remaining_tasks = {"Sat-0": 3, "Sat-1": 3, "Sat-2": 6, "Sat-3": 3}

    def reset(self, *, seed=None, options=None):
        self._step_count = 0
        out = super().reset(seed=seed, options=options) if "seed" in super().reset.__code__.co_varnames else super().reset()
        if isinstance(out, tuple) and len(out) == 2:
            obs, info = out
        else:
            obs, info = out, {}
        self.remaining_tasks = {"Sat-0": 3, "Sat-1": 3, "Sat-2": 6, "Sat-3": 3}
        return obs, info

    def step(self, action_dict: Dict[str, Any]):
        self._step_count += 1
        obs, rews, terms, truncs, infos = super().step(action_dict)

        for agent, r in rews.items():
            if r > 0:
                self.remaining_tasks[agent] = max(0, self.remaining_tasks.get(agent, 0) - 1)

        horizon_reached = self._step_count >= self.max_episode_steps
        no_agents_left = len(self.agents) == 0

        all_keys = set(rews.keys()) | set(terms.keys()) | set(truncs.keys()) | set(self.agents)
        all_done_per_agent = all(terms.get(a, False) or truncs.get(a, False) for a in all_keys) if all_keys else True

        terms["__all__"] = no_agents_left or all_done_per_agent
        truncs["__all__"] = (horizon_reached and not terms["__all__"])

        return obs, rews, terms, truncs, infos

def train_td3_her_dwc():
    sat_args = {
        "imageAttErrorRequirement": 0.01,
        "imageRateErrorRequirement": 0.01,
        "batteryStorageCapacity": 1e9,
        "storedCharge_Init": 1e9,
        "dataStorageCapacity": 1e12,
        "u_max": 0.4,
        "K1": 0.25,
        "K3": 3.0,
        "omega_max": 0.087,
        "servo_Ki": 5.0,
        "servo_P": 150 / 5,
    }
    sat_arg_randomizer = walker_delta_args(altitude=800.0, inc=60.0, n_planes=1)
    
    max_episode_steps = 64
    satellites = [AdvancedImagingSatellite(f"Sat-{i}", sat_args) for i in range(100)]

    for sat in satellites:
        _tune_access_generation(sat, initial=1800.0, step=120.0, max_dur=3600.0)

    env = CustomConstellationTasking(
        satellites=satellites,
        scenario=TargetAreas(n_targets=1000),
        rewarder=CustomUniqueImageReward(),
        communicator=LOSCommunication(),
        sat_arg_randomizer=sat_arg_randomizer,
        log_level="INFO",
        max_episode_steps=max_episode_steps,
    )
    
    obs0, _ = env.reset()
    first_agent = env.agents[0]
    
    if hasattr(env.observation_space(first_agent), 'spaces'):
        state_dim = sum([space.shape[0] for space in env.observation_space(first_agent).spaces.values()])
    else:
        state_dim = env.observation_space(first_agent).shape[0]
    
    action_dims = []
    if hasattr(env.action_space(first_agent), 'spaces'):
        for space in env.action_space(first_agent).spaces:
            if hasattr(space, 'n'):
                action_dims.append(space.n)
            else:
                action_dims.append(space.shape[0])
    else:
        if hasattr(env.action_space(first_agent), 'n'):
            action_dims = [env.action_space(first_agent).n]
        else:
            action_dims = [env.action_space(first_agent).shape[0]]
    
    print(f"State dimension: {state_dim}, Action dimensions: {action_dims}")
    
    agents = {}
    for agent_id in env.agents:
        agents[agent_id] = TD3_HER_DWC(
            state_dim=state_dim,
            action_dims=action_dims,
            device='cuda' if torch.cuda.is_available() else 'cpu',
            her_ratio=0.8
        )
    
    num_episodes = 1000
    max_timesteps = max_episode_steps
    batch_size = 256
    
    rewards_history = []
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        episode_steps = 0
        
        for t in range(max_timesteps):
            action = {}
            for agent_id in env.agents:
                agent_state = state[agent_id]
                if isinstance(agent_state, dict):
                    agent_state = np.concatenate([v.flatten() for v in agent_state.values()])
                action[agent_id] = agents[agent_id].select_action(agent_state, explore=True)
            
            next_state, reward, done, truncated, info = env.step(action)
            
            for agent_id in env.agents:
                agent_state = state[agent_id]
                if isinstance(agent_state, dict):
                    agent_state = np.concatenate([v.flatten() for v in agent_state.values()])
                
                next_agent_state = next_state[agent_id]
                if isinstance(next_agent_state, dict):
                    next_agent_state = np.concatenate([v.flatten() for v in next_agent_state.values()])
                
                agents[agent_id].replay_buffer.add(
                    state=agent_state,
                    action=np.array(action[agent_id]),
                    reward=reward[agent_id],
                    next_state=next_agent_state,
                    done=done[agent_id]
                )
            
            for agent_id in env.agents:
                agents[agent_id].train(batch_size)
            
            state = next_state
            episode_reward += sum(reward.values())
            episode_steps += 1
            
            if all(done.values()):
                break
        
        rewards_history.append(episode_reward)
        
        if episode % 10 == 0:
            print(f"Episode {episode}, Reward: {episode_reward:.2f}, Steps: {episode_steps}")
        
        if episode % 100 == 0:
            for agent_id in env.agents:
                agents[agent_id].save(f"td3_her_dwc_{agent_id}_checkpoint_{episode}.pth")
    
    for agent_id in env.agents:
        agents[agent_id].save(f"td3_her_dwc_{agent_id}_final.pth")
    env.close()
    
    return rewards_history

if __name__ == "__main__":
    rewards = train_td3_her_dwc()
    print("Training completed!")

2025-10-03 22:40:15,362 gym                            INFO       Resetting environment with seed=2141725512
2025-10-03 22:40:15,366 scene.targets                  INFO       Generating 100 targets
2025-10-03 22:40:15,845 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 0.00 to 600.00 seconds
2025-10-03 22:40:15,858 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 600.00 to 1200.00 seconds
2025-10-03 22:40:15,872 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 1200.00 to 1800.00 seconds
2025-10-03 22:40:15,883 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 1800.00 to 2400.00 seconds
2025-10-03 22:40:15,893 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 2400.00 to 3000.00 seconds
2025-10-03 22:40:15,904 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows

State dimension: 10, Action dimensions: [np.int64(9)]


2025-10-03 22:40:19,059 gym                            INFO       Resetting environment with seed=1544281916
2025-10-03 22:40:19,064 scene.targets                  INFO       Generating 100 targets
2025-10-03 22:40:21,625 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 0.00 to 600.00 seconds
2025-10-03 22:40:21,639 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 600.00 to 1200.00 seconds
2025-10-03 22:40:21,653 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 1200.00 to 1800.00 seconds
2025-10-03 22:40:21,668 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 1800.00 to 2400.00 seconds
2025-10-03 22:40:21,680 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 2400.00 to 3000.00 seconds
2025-10-03 22:40:21,692 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows

Episode 0, Reward: 26.16, Steps: 64
Training completed!


In [8]:
from Basilisk.architecture import messaging

print("\n=== TESTING PHASE (With Fault Injection at Step 8) ===")
sat_args = {
    "imageAttErrorRequirement": 0.01,
    "imageRateErrorRequirement": 0.01,
    "batteryStorageCapacity": 1e9,
    "storedCharge_Init": 1e9,
    "dataStorageCapacity": 1e12,
    "u_max": 0.4,
    "K1": 0.25,
    "K3": 3.0,
    "omega_max": 0.087,
    "servo_Ki": 5.0,
    "servo_P": 150 / 5,
}
sat_arg_randomizer = walker_delta_args(altitude=800.0, inc=60.0, n_planes=1)

def test_env_creator(env_config):
    max_episode_steps = env_config.get("max_episode_steps", 64)
    satellites = [AdvancedImagingSatellite(f"Sat-{i}", sat_args) for i in range(4)]

    for sat in satellites:
        _tune_access_generation(sat, initial=1800.0, step=120.0, max_dur=3600.0)

    return CustomConstellationTasking(
        satellites=satellites,
        scenario=TargetAreas(n_targets=12),  
        rewarder=CustomUniqueImageReward(),
        communicator=LOSCommunication(),
        sat_arg_randomizer=sat_arg_randomizer,
        log_level="INFO",
        max_episode_steps=max_episode_steps,
    )

test_env = test_env_creator({})

obs0, _ = test_env.reset()
first_agent = test_env.agents[0]

if hasattr(test_env.observation_space(first_agent), 'spaces'):
    state_dim = sum([space.shape[0] for space in test_env.observation_space(first_agent).spaces.values()])
else:
    state_dim = test_env.observation_space(first_agent).shape[0]

action_dims = []
if hasattr(test_env.action_space(first_agent), 'spaces'):
    for space in test_env.action_space(first_agent).spaces:
        if hasattr(space, 'n'):
            action_dims.append(space.n)
        else:
            action_dims.append(space.shape[0])
else:
    if hasattr(test_env.action_space(first_agent), 'n'):
        action_dims = [test_env.action_space(first_agent).n]
    else:
        action_dims = [test_env.action_space(first_agent).shape[0]]

print(f"Test Environment - State dimension: {state_dim}, Action dimensions: {action_dims}")

agents = {}
for agent_id in ["Sat-0", "Sat-1", "Sat-2", "Sat-3"]:
    agents[agent_id] = TD3_HER_DWC(
        state_dim=state_dim,
        action_dims=action_dims,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        her_ratio=0.8
    )
    try:
        agents[agent_id].load(f"td3_her_dwc_{agent_id}_final.pth")
        print(f"Loaded model for {agent_id}")
    except FileNotFoundError:
        print(f"Warning: No saved model found for {agent_id}, using untrained agent")

def inject_failure(sat_index: int):
    """Force a satellite 'dead' by zeroing power each step."""
    sat = test_env.unwrapped.satellites[sat_index]
    def isnt_alive(log_failure=False, _sat=sat):
        death_message = messaging.PowerStorageStatusMsgPayload()
        death_message.storageLevel = 0.0
        _sat.dynamics.powerMonitor.batPowerOutMsg.write(death_message)
        return _sat.dynamics.is_alive(log_failure=log_failure) and _sat.fsw.is_alive(log_failure=log_failure)
    sat.is_alive = isnt_alive

def find_nearest_satellite(faulty_sat_index):
    """Find the nearest operational satellite to transfer tasks to."""
    operational_sats = []
    for i, sat in enumerate(test_env.unwrapped.satellites):
        if i != faulty_sat_index and sat.name in test_env.agents:
            operational_sats.append(i)
    
    if operational_sats:
        return operational_sats[0]  
    return None

max_steps = 45
max_wallclock_s = 180

observations, _ = test_env.reset()
test_env.remaining_tasks = {"Sat-0": 3, "Sat-1": 3, "Sat-2": 3, "Sat-3": 3}

current_step = 0
episode_reward = 0.0
t0 = time.time()

print(f"Initial state (Step {current_step}):")
for sat_name in ["Sat-0", "Sat-1", "Sat-2", "Sat-3"]:
    imaged = getattr(test_env.rewarder, "imaged_by_sat", {}).get(sat_name, 0)
    remaining = test_env.remaining_tasks.get(sat_name, 0)
    print(f"{sat_name}: imaged = {imaged} tasks = {imaged + remaining} remaining = {remaining}")

performance_metrics = {
    'steps': [],
    'rewards': [],
    'tasks_completed': [],
    'satellites_active': []
}

while current_step < max_steps and test_env.agents:
    if time.time() - t0 > max_wallclock_s:
        print(f"[Test] Wallclock timeout ({max_wallclock_s}s). Breaking.")
        break

    if current_step == 8 and "Sat-1" in test_env.agents:
        print(f"\n=== INJECTING FAULT AT STEP {current_step} ===")
        print("Before fault injection:")
        for sat_name in ["Sat-0", "Sat-1", "Sat-2", "Sat-3"]:
            imaged = getattr(test_env.rewarder, "imaged_by_sat", {}).get(sat_name, 0)
            remaining = test_env.remaining_tasks.get(sat_name, 0)
            print(f"{sat_name}: imaged = {imaged} tasks = {imaged + remaining} remaining = {remaining}")
        
        nearest_sat_index = find_nearest_satellite(1)  
        if nearest_sat_index is not None:
            nearest_sat_name = f"Sat-{nearest_sat_index}"
            sat1_tasks = test_env.remaining_tasks.get("Sat-1", 0)
            test_env.remaining_tasks[nearest_sat_name] += sat1_tasks
            test_env.remaining_tasks["Sat-1"] = 0
            
            print(f"\nTransferring {sat1_tasks} tasks from Sat-1 to {nearest_sat_name}")
            print("After fault injection and task transfer:")
            for sat_name in ["Sat-0", "Sat-1", "Sat-2", "Sat-3"]:
                imaged = getattr(test_env.rewarder, "imaged_by_sat", {}).get(sat_name, 0)
                remaining = test_env.remaining_tasks.get(sat_name, 0)
                print(f"{sat_name}: imaged = {imaged} tasks = {imaged + remaining} remaining = {remaining}")
        
        inject_failure(1)
        print(f"\n*** FAULT INJECTED: Sat-1 is now faulty ***")
        print("=================================\n")

    actions = {}
    for agent in test_env.agents:
        if agent in agents:
            agent_state = observations[agent]
            if isinstance(agent_state, dict):
                agent_state = np.concatenate([v.flatten() for v in agent_state.values()])
            
            action = agents[agent].select_action(agent_state, explore=False)
            actions[agent] = action
        else:
            actions[agent] = [0] * len(action_dims)

    try:
        observations, rewards, terminations, truncations, infos = test_env.step(actions)
        episode_reward += sum(rewards.values())
        
        performance_metrics['steps'].append(current_step)
        performance_metrics['rewards'].append(sum(rewards.values()))
        performance_metrics['tasks_completed'].append(
            sum(getattr(test_env.rewarder, "imaged_by_sat", {}).values())
        )
        performance_metrics['satellites_active'].append(len(test_env.agents))
        
    except KeyError as e:
        print(f"Warning: KeyError encountered at step {current_step}: {e}")
        print("This is likely due to the fault injection. Continuing simulation...")
        if "Sat-1" in observations:
            del observations["Sat-1"]
        rewards = {agent: 0.0 for agent in test_env.agents}

    current_step += 1

    if current_step % 5 == 0:
        print(f"Step {current_step}: Reward this step = {sum(rewards.values()):.2f}, "
              f"Total reward = {episode_reward:.2f}, Active satellites = {len(test_env.agents)}")

print(f"\n=== TEST RESULTS ===")
print(f"Test episode reward: {episode_reward:.2f}")
print(f"Total steps completed: {current_step}")






test_env.close()
print("\n=== TESTING COMPLETED ===")

2025-10-04 07:09:14,897                                WARNING    Creating logger for new env on PID=21308. Old environments in process may now log times incorrectly.
2025-10-04 07:09:14,902 gym                            INFO       Resetting environment with seed=169996956
2025-10-04 07:09:14,914 scene.targets                  INFO       Generating 12 targets



=== TESTING PHASE (With Fault Injection at Step 8) ===


2025-10-04 07:09:15,454 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 0.00 to 600.00 seconds
2025-10-04 07:09:15,463 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 600.00 to 1200.00 seconds
2025-10-04 07:09:15,470 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 1200.00 to 1800.00 seconds
2025-10-04 07:09:15,481 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 1800.00 to 2400.00 seconds
2025-10-04 07:09:15,489 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 2400.00 to 3000.00 seconds
2025-10-04 07:09:15,501 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 3000.00 to 3600.00 seconds
2025-10-04 07:09:15,515 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 3600.00 to 4200.00 seconds
2025-10-04 07:09:15,525 s

Test Environment - State dimension: 10, Action dimensions: [np.int64(9)]
Loaded model for Sat-0
Loaded model for Sat-1
Loaded model for Sat-2
Loaded model for Sat-3


2025-10-04 07:09:21,269 gym                            INFO       Resetting environment with seed=2242020960
2025-10-04 07:09:21,273 scene.targets                  INFO       Generating 12 targets
2025-10-04 07:09:35,139 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 0.00 to 600.00 seconds
2025-10-04 07:09:35,149 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 600.00 to 1200.00 seconds
2025-10-04 07:09:35,162 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 1200.00 to 1800.00 seconds
2025-10-04 07:09:35,175 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 1800.00 to 2400.00 seconds
2025-10-04 07:09:35,189 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows from 2400.00 to 3000.00 seconds
2025-10-04 07:09:35,202 sats.satellite.Sat-0           INFO       <0.00> Sat-0: Finding opportunity windows 

...

Step 20: Reward this step = 0.77, Total reward = 6.65, Active satellites = 3


2025-10-04 07:11:34,707 sats.satellite.Sat-0           INFO       <142225.00> Sat-0: imaged Target(SA_Target_6)
2025-10-04 07:11:34,745 sats.satellite.Sat-0           INFO       <142225.00> Sat-0: Satellite Sat-0 requires retasking
2025-10-04 07:11:34,749 sats.satellite.Sat-2           INFO       <142225.00> Sat-2: Finding opportunity windows from 145800.00 to 146400.00 seconds
2025-10-04 07:11:34,774 sats.satellite.Sat-2           INFO       <142225.00> Sat-2: Finding opportunity windows from 146400.00 to 147000.00 seconds
2025-10-04 07:11:34,798 sats.satellite.Sat-2           INFO       <142225.00> Sat-2: Finding opportunity windows from 147000.00 to 147600.00 seconds
2025-10-04 07:11:34,822 sats.satellite.Sat-2           INFO       <142225.00> Sat-2: Finding opportunity windows from 147600.00 to 148200.00 seconds
2025-10-04 07:11:34,849 sats.satellite.Sat-2           INFO       <142225.00> Sat-2: Finding opportunity windows from 148200.00 to 148800.00 seconds
2025-10-04 07:11:34,882

Step 25: Reward this step = 0.27, Total reward = 9.18, Active satellites = 3


2025-10-04 07:12:23,365 sats.satellite.Sat-0           INFO       <174152.00> Sat-0: imaged Target(SA_Target_11)
2025-10-04 07:12:23,367 sats.satellite.Sat-0           INFO       <174152.00> Sat-0: Satellite Sat-0 requires retasking
2025-10-04 07:12:23,371 sats.satellite.Sat-0           INFO       <174152.00> Sat-0: Finding opportunity windows from 226800.00 to 227400.00 seconds
2025-10-04 07:12:23,426 sats.satellite.Sat-0           INFO       <174152.00> Sat-0: Finding opportunity windows from 227400.00 to 228000.00 seconds
2025-10-04 07:12:23,468 sats.satellite.Sat-0           INFO       <174152.00> Sat-0: Finding opportunity windows from 228000.00 to 228600.00 seconds
2025-10-04 07:12:23,518 sats.satellite.Sat-0           INFO       <174152.00> Sat-0: Finding opportunity windows from 228600.00 to 229200.00 seconds
2025-10-04 07:12:23,568 sats.satellite.Sat-0           INFO       <174152.00> Sat-0: Finding opportunity windows from 229200.00 to 229800.00 seconds
2025-10-04 07:12:23,61

[Test] Wallclock timeout (180s). Breaking.

=== TEST RESULTS ===
Test episode reward: 9.78
Total steps completed: 27

=== TESTING COMPLETED ===
